# Importación de librerías

In [1]:
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import filters_and_features as ff
import data_processor as dp
import numpy as np

# Importación de datos

In [2]:
df = dp.lectura()

Se crea el dataframe df_block

In [3]:
df_block = dp.create_df_block(df)

In [6]:
del df

Se limpia y visualiza el contenido

In [ ]:
dp.balance(df_block)
df_block['stimulus'].value_counts()

# Filtro

In [ ]:
df_block_filtered = dp.filtrar(df_block)

In [8]:
del df_block

# Extracción de características

In [ ]:
df_block_features = dp.gen_carac(df_block_filtered)
# del df_block_filtered

In [ ]:
df_block_features.describe()

# Normalización

In [10]:
df_block_normalized = dp.normalizar(df_block_features)
# del df_block_features

In [ ]:
df_block_normalized.describe()

# Creación de dataframes de entrenamiento y testeo

In [12]:
y_block = df_block_normalized.iloc[:, -1]
X_block = df_block_normalized.iloc[:, :-1]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_block, y_block,
                                                    random_state=100,
                                                    test_size=0.30,
                                                    shuffle=True)

# del df_block_normalized

In [ ]:
print(X_train_b.shape,y_train_b.shape)

# Entrenamiento y testeo del modelo

Importamos librerías

In [24]:
# Random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Rotation forest
from sktime.classification.sklearn import RotationForest
from sktime.datasets import load_unit_test
from sktime.datatypes._panel._convert import from_nested_to_3d_numpy

# Artificial Neural Networks (ANN)
import setuptools.dist
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Hidden Markov model (HMM)
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder

Random forest

In [20]:
rf_b = RandomForestClassifier(
    max_depth=20,             # Profundidad máxima de los árboles
    criterion='entropy',      # Criterio de medida para la calidad de la división ('entropy' o 'gini')
    min_samples_split=4,      # Número mínimo de muestras requeridas para dividir un nodo
    random_state=99,          # Para reproducibilidad del modelo
    n_estimators=3000,        # Número de árboles en el bosque
    verbose=1,                # Imprimir información durante el entrenamiento
    oob_score=True,           # Calcular la precisión del modelo fuera de la bolsa (OOB)
    n_jobs=-1                 # Usar todos los procesadores disponibles para acelerar el entrenamiento
)
rf_b.fit(X_train_b, y_train_b)
y_pred_b_RanF = rf_b.predict(X_test_b)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 418 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 768 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 1218 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done 1768 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done 2418 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done 3000 out of 3000 | elapsed:    3.8s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 418 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 768 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 1218 tasks      | elapsed:    0.0s
[Parallel(n_job

Rotation forest

In [28]:
clf = RotationForest(n_estimators=10)
clf.fit(X_train_b, y_train_b)
y_pred_b_RotF = clf.predict(X_test_b)

Artificial Neural Networks (ANN)

In [37]:
le = LabelEncoder()
y_train_b_enc = le.fit_transform(y_train_b)  # Transforma las etiquetas a valores consecutivos (0 a 7)
y_test_b_enc = le.transform(y_test_b)

# Asegurarse de que el número de clases sea correcto
num_classes = len(np.unique(y_train_b_enc))
print("Número de clases detectadas en y_train_b:", num_classes)  # Esto debería ser 8

# Convertir a formato categórico
y_train_b_cat = to_categorical(y_train_b_enc, num_classes=num_classes)
y_test_b_cat = to_categorical(y_test_b_enc, num_classes=num_classes)

# Definir y entrenar la red neuronal
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_b.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')  # La capa de salida ahora usa 8 neuronas, una por clase
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_b, y_train_b_cat, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Predicción y decodificación
y_pred_probs = model.predict(X_test_b)
y_pred_ann_e = np.argmax(y_pred_probs, axis=1)
y_pred_b_ann = le.inverse_transform(y_pred_ann_e)  # Convertir a las etiquetas originales

Número de clases detectadas en y_train_b: 8
Epoch 1/50


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2639 - loss: 2.1277 - val_accuracy: 0.5203 - val_loss: 1.2823
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5887 - loss: 1.1574 - val_accuracy: 0.6892 - val_loss: 0.8929
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7320 - loss: 0.8364 - val_accuracy: 0.7770 - val_loss: 0.6767
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8213 - loss: 0.5585 - val_accuracy: 0.8446 - val_loss: 0.5368
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8418 - loss: 0.4600 - val_accuracy: 0.8176 - val_loss: 0.5102
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8148 - loss: 0.4749 - val_accuracy: 0.8649 - val_loss: 0.4385
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8853 - loss: 0.3414 - val_accuracy: 0.8581 - val_loss: 0.4251
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9001 - loss: 0.2905 - val_accuracy: 0.8784 - val_loss: 0.4009
Epo

# Resultados

Random forest

In [30]:
confusion_matrix(y_pred_b_RanF, y_test_b)

array([[42,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 36,  0,  0,  0,  0,  0,  0],
       [ 0,  0, 29,  3,  2,  0,  0,  0],
       [ 0,  0,  4, 39,  2,  0,  1,  0],
       [ 0,  0,  8,  0, 37,  0,  0,  0],
       [ 0,  0,  0,  1,  0, 31,  0,  1],
       [ 0,  0,  0,  0,  0,  0, 41,  0],
       [ 0,  2,  0,  0,  0,  2,  0, 36]])

In [31]:
accuracy_score(y_pred_b_RanF, y_test_b)

0.917981072555205

In [32]:
print(classification_report(y_pred_b_RotF, y_test_b))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        42
           1       0.89      0.94      0.92        36
           3       0.68      0.76      0.72        37
           4       0.86      0.86      0.86        43
           6       0.88      0.80      0.84        45
           9       0.97      1.00      0.98        32
          10       1.00      1.00      1.00        42
          11       0.97      0.90      0.94        40

    accuracy                           0.91       317
   macro avg       0.91      0.91      0.91       317
weighted avg       0.91      0.91      0.91       317



Rotation forest

In [33]:
confusion_matrix(y_pred_b_RotF, y_test_b)

array([[42,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 34,  0,  1,  0,  0,  0,  1],
       [ 0,  1, 28,  3,  5,  0,  0,  0],
       [ 0,  0,  6, 37,  0,  0,  0,  0],
       [ 0,  1,  7,  1, 36,  0,  0,  0],
       [ 0,  0,  0,  0,  0, 32,  0,  0],
       [ 0,  0,  0,  0,  0,  0, 42,  0],
       [ 0,  2,  0,  1,  0,  1,  0, 36]])

In [34]:
accuracy_score(y_pred_b_RotF, y_test_b)

0.9053627760252366

In [35]:
print(classification_report(y_pred_b_RotF, y_test_b))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        42
           1       0.89      0.94      0.92        36
           3       0.68      0.76      0.72        37
           4       0.86      0.86      0.86        43
           6       0.88      0.80      0.84        45
           9       0.97      1.00      0.98        32
          10       1.00      1.00      1.00        42
          11       0.97      0.90      0.94        40

    accuracy                           0.91       317
   macro avg       0.91      0.91      0.91       317
weighted avg       0.91      0.91      0.91       317



Artificial Neural Networks (ANN)

In [38]:
confusion_matrix(y_pred_b_ann, y_test_b)

array([[42,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 37,  0,  0,  0,  0,  0,  0],
       [ 0,  0, 35,  1,  1,  0,  0,  0],
       [ 0,  1,  1, 41,  2,  0,  0,  0],
       [ 0,  0,  5,  1, 38,  0,  0,  0],
       [ 0,  0,  0,  0,  0, 31,  0,  0],
       [ 0,  0,  0,  0,  0,  0, 42,  0],
       [ 0,  0,  0,  0,  0,  2,  0, 37]])

In [39]:
accuracy_score(y_pred_b_ann, y_test_b)

0.9558359621451105

In [40]:
print(classification_report(y_pred_b_ann, y_test_b))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        42
           1       0.97      1.00      0.99        37
           3       0.85      0.95      0.90        37
           4       0.95      0.91      0.93        45
           6       0.93      0.86      0.89        44
           9       0.94      1.00      0.97        31
          10       1.00      1.00      1.00        42
          11       1.00      0.95      0.97        39

    accuracy                           0.96       317
   macro avg       0.96      0.96      0.96       317
weighted avg       0.96      0.96      0.96       317

